# Inception Model


<img src="D:/codes-major/Inception-v3-Model-image.png" alt="Lung Disease Classification" width="1000" height="500">


### inception model code 

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

# Dataset path
data_dir = "D:/1/Dataset-lung"

# Parameters
img_height, img_width = 299, 299  # Required input size for InceptionV3
batch_size = 32
num_classes = None  # To be determined based on data generators
epochs = 20
learning_rate = 0.0001

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Data augmentation and preprocessing
datagen_train = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # Splitting dataset into training and validation
)

datagen_val = ImageDataGenerator(rescale=1.0/255, validation_split=0.2)

train_generator = datagen_train.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen_val.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Determine the number of classes
num_classes = len(train_generator.class_indices)

# Load pre-trained InceptionV3 model without the top layer
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))

# Adding custom layers on top
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)  # Dropout for regularization
x = Dense(1024, activation='relu')(x)
outputs = Dense(num_classes, activation='softmax')(x)

# Define the model
model = Model(inputs=base_model.input, outputs=outputs)

# Freeze the base_model layers for transfer learning
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer=Adam(learning_rate=learning_rate),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator,
    callbacks=[early_stopping, reduce_lr]
)

# Unfreeze some base_model layers for fine-tuning
for layer in base_model.layers[-30:]:  # Unfreezing the last 30 layers
    layer.trainable = True

# Re-compile the model for fine-tuning
model.compile(optimizer=Adam(learning_rate=learning_rate/10),  # Lower learning rate
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Fine-tune the model
history_fine_tune = model.fit(
    train_generator,
    epochs=epochs//2,
    validation_data=val_generator,
    callbacks=[early_stopping, reduce_lr]
)

# Save the model
model.save("inceptionv3_lung_disease_classifier.h5")

print("Model training complete and saved!")


Found 2856 images belonging to 4 classes.
Found 713 images belonging to 4 classes.


C:\Users\ganes\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.6576 - loss: 0.8713   

C:\Users\ganes\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 570s 6s/step - accuracy: 0.6587 - loss: 0.8687 - val_accuracy: 0.7574 - val_loss: 0.6146 - learning_rate: 1.0000e-04
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 585s 6s/step - accuracy: 0.8440 - loss: 0.4138 - val_accuracy: 0.7616 - val_loss: 0.5835 - learning_rate: 1.0000e-04
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 586s 6s/step - accuracy: 0.8693 - loss: 0.3373 - val_accuracy: 0.7616 - val_loss: 0.6080 - learning_rate: 1.0000e-04
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 551s 6s/step - accuracy: 0.8730 - loss: 0.3301 - val_accuracy: 0.8401 - val_loss: 0.4224 - learning_rate: 1.0000e-04
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 542s 6s/step - accuracy: 0.8964 - loss: 0.2698 - val_accuracy: 0.7714 - val_loss: 0.5798 - learning_rate: 1.0000e-04
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 564s 6s/step - accuracy: 0.8937 - loss: 0.2930 - val_accuracy: 0.8036 - val_loss: 0.4755 - learning_rate: 1.0000e-04
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 559s 6s/step - accuracy: 0.8972 - loss: 0.2

Model training complete and saved!


### summery of the model 

In [2]:
from tensorflow.keras.models import load_model

# Load the model
model = load_model("inceptionv3_lung_disease_classifier.h5")

# Verify the model structure
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 299, 299, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 149, 149, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 149, 149, 32)      │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 149, 149, 32)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 147, 147, 32)      │           9,216 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 147, 147, 32)      │              96 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 147, 147, 32)      │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 147, 147, 64)      │          18,432 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 147, 147, 64)      │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 147, 147, 64)      │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 73, 73, 64)        │               0 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 73, 73, 80)        │           5,120 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 73, 73, 80)        │             240 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 73, 73, 80)        │               0 │ batch_normalization_3[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 71, 71, 192)       │         138,240 │ activation_3[0][0]         │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 23,905,062 (91.19 MB)

 Trainable params: 7,258,308 (27.69 MB)

 Non-trainable params: 16,646,752 (63.50 MB)

 Optimizer params: 2 (12.00 B)

In [4]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the trained model
model = load_model("inceptionv3_lung_disease_classifier.h5")

# Define the class labels manually
# Replace these with the actual labels from your dataset (in order of the indices used during training)
class_labels = ["COVID", "Normal", "Tuberculosis", "Viral Pneumonia"]

# Function to preprocess an image for prediction
def preprocess_image(img_path, target_size=(299, 299)):
    # Load the image with the specified target size
    img = image.load_img(img_path, target_size=target_size)
    # Convert the image to a numpy array
    img_array = image.img_to_array(img)
    # Add batch dimension and normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

# Predict function
def predict_image(img_path):
    # Preprocess the image
    input_image = preprocess_image(img_path)
    # Get predictions
    predictions = model.predict(input_image)
    # Decode the prediction
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_label = class_labels[predicted_class_index]
    predicted_confidence = predictions[0][predicted_class_index]
    return predicted_label, predicted_confidence

# Test prediction
img_path = r"D:\Dataset-lung\Tuberculosis\Tuberculosis-15.png"  # Replace with your test image path
predicted_label, confidence = predict_image(img_path)

print(f"Predicted Label: {predicted_label}")
print(f"Confidence: {confidence:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted Label: Tuberculosis
Confidence: 0.91


In [9]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the trained model
model = load_model("inceptionv3_lung_disease_classifier.h5")

# Define the class labels manually
# Replace these with the actual labels from your dataset (in order of the indices used during training)
class_labels = ["COVID", "Normal", "Tuberculosis", "Viral Pneumonia"]

# Function to preprocess an image for prediction
def preprocess_image(img_path, target_size=(299, 299)):
    # Load the image with the specified target size
    img = image.load_img(img_path, target_size=target_size)
    # Convert the image to a numpy array
    img_array = image.img_to_array(img)
    # Add batch dimension and normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

# Predict function
def predict_image(img_path):
    # Preprocess the image
    input_image = preprocess_image(img_path)
    # Get predictions
    predictions = model.predict(input_image)
    # Decode the prediction
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_label = class_labels[predicted_class_index]
    predicted_confidence = predictions[0][predicted_class_index]
    return predicted_label, predicted_confidence

# Test prediction
img_path = r"D:\Dataset-lung\COVID\COVID-15.png"  # Replace with your test image path
predicted_label, confidence = predict_image(img_path)

print(f"Predicted Label: {predicted_label}")
print(f"Confidence: {confidence:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted Label: COVID
Confidence: 0.56


In [12]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the trained model
model = load_model("inceptionv3_lung_disease_classifier.h5")

# Define the class labels manually
# Replace these with the actual labels from your dataset (in order of the indices used during training)
class_labels = ["COVID", "Normal", "Tuberculosis", "Viral Pneumonia"]

# Function to preprocess an image for prediction
def preprocess_image(img_path, target_size=(299, 299)):
    # Load the image with the specified target size
    img = image.load_img(img_path, target_size=target_size)
    # Convert the image to a numpy array
    img_array = image.img_to_array(img)
    # Add batch dimension and normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

# Predict function
def predict_image(img_path):
    # Preprocess the image
    input_image = preprocess_image(img_path)
    # Get predictions
    predictions = model.predict(input_image)
    # Decode the prediction
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_label = class_labels[predicted_class_index]
    predicted_confidence = predictions[0][predicted_class_index]
    return predicted_label, predicted_confidence

# Test prediction
img_path = r"D:\Dataset-lung\Normal\Normal-13.png"  # Replace with your test image path
predicted_label, confidence = predict_image(img_path)

print(f"Predicted Label: {predicted_label}")
print(f"Confidence: {confidence:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted Label: Normal
Confidence: 0.99
